## **Install Libraries**

In [3]:
%pip install -q presidio-analyzer presidio-anonymizer spacy
%pip install -q https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.7.1/en_core_web_lg-3.7.1-py3-none-any.whl

StatementMeta(, 8ca6c674-8790-4df0-83c1-896f6152e2e6, 18, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



## **Import Libraries**

In [4]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("presidio-analyzer").setLevel(logging.ERROR)

import pandas as pd
import numpy as np
import re
import os

from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine


StatementMeta(, 8ca6c674-8790-4df0-83c1-896f6152e2e6, 20, Finished, Available, Finished, False)

## **Initialize Presidio**

In [3]:
analyzer = AnalyzerEngine()

nric_pattern = Pattern(
    name="Malaysia NRIC Pattern",
    regex=r"\b\d{6}-?\d{2}-?\d{4}\b",
    score=0.95
)

nric_recognizer = PatternRecognizer(
    supported_entity="MY_NRIC",
    patterns=[nric_pattern]
)

malaysia_phone_pattern = Pattern(
    name="Malaysia Phone Pattern",
    regex=r"\b(\+?60|0)?1[0-9]{8,9}\b",
    score=0.85
)

malaysia_phone_recognizer = PatternRecognizer(
    supported_entity="MY_PHONE_NUMBER",
    patterns=[malaysia_phone_pattern]
)

analyzer.registry.add_recognizer(nric_recognizer)
analyzer.registry.add_recognizer(malaysia_phone_recognizer)

StatementMeta(, 68b2d141-3696-4c17-9191-d3978c91bd15, 11, Finished, Available, Finished, False)

## **Read Raw Datasets from Lakehouse**

In [4]:
data_owner = "owner_demo"

raw_path = f"/lakehouse/default/Files/raw/{data_owner}/"

datasets = {
    "customer": pd.read_csv(raw_path + "customer.csv"),
    "billing": pd.read_csv(raw_path + "billing.csv"),
    "service": pd.read_csv(raw_path + "service.csv")
}

for name, df in datasets.items():
    print(f"Loaded: {name} | Shape: {df.shape}")

StatementMeta(, 68b2d141-3696-4c17-9191-d3978c91bd15, 12, Finished, Available, Finished, False)

Loaded: customer | Shape: (1200, 24)
Loaded: billing | Shape: (2500, 12)
Loaded: service | Shape: (1500, 16)


## **Cleaning Functions**

In [6]:
def generic_cleaning(df):
    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

    duplicate_count = df.duplicated().sum()
    df = df.drop_duplicates()

    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()

    missing_values = ["", "nan", "none", "null", "na", "n/a", "-", "--"]
    df = df.replace(missing_values, np.nan)

    for col in df.columns:
        if "date" in col.lower() or "time" in col.lower():
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

    for col in df.columns:
        if df[col].dtype == "object":
            converted = pd.to_numeric(df[col], errors="coerce")
            numeric_ratio = converted.notnull().sum() / len(df) if len(df) > 0 else 0

            if numeric_ratio > 0.85:
                df[col] = converted

    return df, duplicate_count


def apply_domain_specific_rules(df):
    df = df.copy()

    if "postcode" in df.columns:
        df["postcode"] = (
            df["postcode"]
            .astype(str)
            .str.replace(".0", "", regex=False)
            .str.zfill(5)
        )
        df["postcode"] = df["postcode"].replace("00nan", np.nan)

    if "phone" in df.columns:
        df["phone"] = (
            df["phone"]
            .astype(str)
            .str.replace(".0", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace("-", "", regex=False)
        )

        df["phone"] = df["phone"].apply(
            lambda x: np.nan if str(x).lower() in ["nan", "none", "null", "na"] else x
        )

        df["phone"] = df["phone"].apply(
            lambda x: "0" + x.lstrip("0") if pd.notnull(x) and not str(x).startswith("+60") else x
        )

    date_cols = [
        "subscription_start_date", "subscription_end_date",
        "billing_period_start", "billing_period_end",
        "service_start_date", "service_end_date"
    ]

    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

    if "age" in df.columns:
        df["age"] = pd.to_numeric(df["age"], errors="coerce").astype("Int64")

    numeric_cols = [
        "subtotal_amount", "discount_amount", "tax_amount", "total_amount",
        "bandwidth_mbps", "monthly_usage_gb",
        "avg_latency_ms", "jitter_ms", "monthly_outages"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def clean_dataset(df):
    df, duplicate_count = generic_cleaning(df)
    df = apply_domain_specific_rules(df)
    return df, duplicate_count

StatementMeta(, 68b2d141-3696-4c17-9191-d3978c91bd15, 14, Finished, Available, Finished, False)

## **Quality Checking Functions**

In [7]:
def is_expected_missing(df, col):
    if "customer_type" not in df.columns:
        return pd.Series(False, index=df.index)

    expected = pd.Series(False, index=df.index)

    individual_expected_null_cols = [
        "company_name", "industry_sector", "sme_flag"
    ]

    business_expected_null_cols = [
        "full_name", "gender", "age", "income", "nric"
    ]

    customer_type = df["customer_type"].astype(str).str.lower()

    if col in individual_expected_null_cols:
        expected = customer_type.eq("individual") & df[col].isnull()

    elif col in business_expected_null_cols:
        expected = customer_type.eq("business") & df[col].isnull()

    return expected


def generate_quality_report(df, dataset_name):
    report = []
    total_rows = len(df)

    for col in df.columns:
        raw_missing_count = df[col].isnull().sum()
        expected_missing_count = is_expected_missing(df, col).sum()
        unexpected_missing_count = raw_missing_count - expected_missing_count

        unexpected_missing_percentage = (
            unexpected_missing_count / total_rows * 100
            if total_rows > 0 else 0
        )

        completeness_score = 100 - unexpected_missing_percentage
        unique_count = df[col].nunique(dropna=True)

        if pd.api.types.is_object_dtype(df[col]) and unique_count <= 30:
            uniqueness_score = 100
        else:
            uniqueness_score = unique_count / total_rows * 100 if total_rows > 0 else 0

        if pd.api.types.is_numeric_dtype(df[col]):
            valid_count = pd.to_numeric(df[col], errors="coerce").notnull().sum()

        elif pd.api.types.is_datetime64_any_dtype(df[col]):
            valid_count = pd.to_datetime(df[col], errors="coerce").notnull().sum()

        else:
            valid_count = df[col].notnull().sum() + expected_missing_count

        validity_score = valid_count / total_rows * 100 if total_rows > 0 else 0

        overall_quality_score = round(
            completeness_score * 0.45 +
            validity_score * 0.45 +
            uniqueness_score * 0.10,
            2
        )

        if overall_quality_score >= 90:
            quality_status = "Excellent"
        elif overall_quality_score >= 75:
            quality_status = "Good"
        elif overall_quality_score >= 60:
            quality_status = "Moderate"
        else:
            quality_status = "Poor"

        report.append({
            "dataset_name": dataset_name,
            "column_name": col,
            "data_type": str(df[col].dtype),
            "total_rows": total_rows,
            "raw_missing_count": raw_missing_count,
            "expected_missing_count": expected_missing_count,
            "unexpected_missing_count": unexpected_missing_count,
            "unique_count": unique_count,
            "completeness_score": round(completeness_score, 2),
            "validity_score": round(validity_score, 2),
            "uniqueness_score": round(uniqueness_score, 2),
            "overall_quality_score": overall_quality_score,
            "quality_status": quality_status
        })

    return pd.DataFrame(report)

StatementMeta(, 68b2d141-3696-4c17-9191-d3978c91bd15, 15, Finished, Available, Finished, False)

## **Governance Policy Configuration**

In [5]:
DIRECT_IDENTIFIER_ENTITIES = [
    "PERSON",
    "EMAIL_ADDRESS",
    "PHONE_NUMBER",
    "MY_PHONE_NUMBER",
    "MY_NRIC",
    "PASSPORT",
    "CREDIT_CARD"
]

QUASI_IDENTIFIER_ENTITIES = [
    "LOCATION",
    "DATE_TIME"
]

FULL_MASK_COLUMNS = [
    "nric",
    "passport",
    "credit_card"
]

PARTIAL_MASK_COLUMNS = [
    "full_name",
    "phone",
    "email",
    "address"
]

PRESERVE_COLUMNS = [
    "customer_id", "service_id", "invoice_id",
    "customer_type", "company_name", "industry_sector",
    "state", "district", "postcode",
    "subscription_plan", "subscription_start_date", "subscription_end_date",
    "status", "payment_method", "billing_cycle",
    "preferred_language", "gender", "age", "income",
    "sme_flag", "kyc_status",
    "billing_period_start", "billing_period_end",
    "pricing_tier", "currency",
    "subtotal_amount", "discount_amount", "tax_amount", "total_amount",
    "service_category", "service_status", "region",
    "access_technology", "router_model", "device_os", "sla_tier",
    "service_start_date", "service_end_date",
    "bandwidth_mbps", "monthly_usage_gb",
    "avg_latency_ms", "jitter_ms", "monthly_outages"
]

StatementMeta(, 68b2d141-3696-4c17-9191-d3978c91bd15, 13, Finished, Available, Finished, False)

## **Privacy Detection and Masking Functions**

In [8]:
def analyze_column_with_presidio(series, sample_size=50):
    sample_values = series.dropna().astype(str).head(sample_size)
    entity_counts = {}

    for value in sample_values:
        results = analyzer.analyze(text=value, language="en")

        for result in results:
            entity = result.entity_type
            entity_counts[entity] = entity_counts.get(entity, 0) + 1

    if len(entity_counts) == 0 or len(sample_values) == 0:
        return "None", 0, 0, {}

    dominant_entity = max(entity_counts, key=entity_counts.get)
    dominant_count = entity_counts[dominant_entity]
    detection_ratio = dominant_count / len(sample_values)

    return dominant_entity, dominant_count, detection_ratio, entity_counts


def determine_sensitivity_and_action(col, dominant_entity, detection_ratio):
    col_lower = col.lower()

    if col_lower in PRESERVE_COLUMNS:
        return "Non-sensitive / Business-useful", "Preserve", "Governance preserve rule"

    if col_lower in FULL_MASK_COLUMNS:
        return "High", "Full Mask", "Governance full-mask rule"

    if col_lower in PARTIAL_MASK_COLUMNS:
        return "High", "Partial Mask", "Governance partial-mask rule"

    if dominant_entity in DIRECT_IDENTIFIER_ENTITIES and detection_ratio >= 0.30:
        return "High", "Partial Mask", "Dynamic Presidio direct identifier detection"

    if dominant_entity in QUASI_IDENTIFIER_ENTITIES and detection_ratio >= 0.50:
        return "Medium", "Preserve", "Quasi-identifier preserved for analytics utility"

    return "Non-sensitive", "Preserve", "No sensitive entity requiring masking"


def detect_privacy_risks(df, dataset_name):
    privacy_report = []

    for col in df.columns:
        dominant_entity, dominant_count, detection_ratio, entity_counts = analyze_column_with_presidio(df[col])

        sensitivity_level, masking_action, governance_reason = determine_sensitivity_and_action(
            col, dominant_entity, detection_ratio
        )

        privacy_report.append({
            "dataset_name": dataset_name,
            "column_name": col,
            "detected_entity": dominant_entity,
            "entity_detection_count": dominant_count,
            "entity_detection_ratio": round(detection_ratio, 2),
            "all_detected_entities": str(entity_counts),
            "sensitivity_level": sensitivity_level,
            "masking_action": masking_action,
            "governance_reason": governance_reason
        })

    return pd.DataFrame(privacy_report)


def partial_mask_email(value):
    if pd.isnull(value):
        return np.nan

    value = str(value)

    if "@" not in value:
        return "**"

    username, domain = value.split("@", 1)
    masked_username = username[:2] + "**" if len(username) > 2 else username[:1] + "**"

    return masked_username + "@" + domain


def partial_mask_phone(value):
    if pd.isnull(value):
        return np.nan

    value = str(value)

    if len(value) >= 6:
        return value[:3] + "**" + value[-2:]

    return "**"


def partial_mask_name(value):
    if pd.isnull(value):
        return np.nan

    value = str(value)

    if len(value) > 0:
        return value[0] + "**"

    return "**"


def generic_partial_mask(value):
    if pd.isnull(value):
        return np.nan

    value = str(value)

    if len(value) >= 4:
        return value[:2] + "**" + value[-1]

    return "**"


def mask_value(value, col, entity, masking_action):
    col_lower = col.lower()

    if pd.isnull(value):
        return np.nan

    if masking_action == "Preserve":
        return value

    if masking_action == "Full Mask":
        return "***"

    if masking_action == "Partial Mask":
        if col_lower == "nric" or entity == "MY_NRIC":
            return "***"

        if col_lower == "email" or entity == "EMAIL_ADDRESS":
            return partial_mask_email(value)

        if col_lower == "phone" or entity in ["PHONE_NUMBER", "MY_PHONE_NUMBER"]:
            return partial_mask_phone(value)

        if col_lower == "full_name" or entity == "PERSON":
            return partial_mask_name(value)

        if col_lower == "address":
            return "**"

        return generic_partial_mask(value)

    return value


def mask_sensitive_data(df, privacy_report):
    df_masked = df.copy()

    for _, row in privacy_report.iterrows():
        col = row["column_name"]
        entity = row["detected_entity"]
        masking_action = row["masking_action"]

        if col in df_masked.columns:
            df_masked[col] = df_masked[col].apply(
                lambda x: mask_value(x, col, entity, masking_action)
            )

    return df_masked

StatementMeta(, 68b2d141-3696-4c17-9191-d3978c91bd15, 16, Finished, Available, Finished, False)

## **Execute Processing Workflow**

In [9]:
cleaned_datasets = {}
masked_datasets = {}

all_quality_reports = []
all_privacy_reports = []
summary_report = []

for dataset_name, df in datasets.items():
    print(f"\nProcessing dataset: {dataset_name}")

    original_rows, original_cols = df.shape

    cleaned_df, duplicate_count = clean_dataset(df)
    quality_report = generate_quality_report(cleaned_df, dataset_name)
    privacy_report = detect_privacy_risks(cleaned_df, dataset_name)
    masked_df = mask_sensitive_data(cleaned_df, privacy_report)

    cleaned_datasets[dataset_name] = cleaned_df
    masked_datasets[dataset_name] = masked_df

    all_quality_reports.append(quality_report)
    all_privacy_reports.append(privacy_report)

    dataset_quality_score = round(quality_report["overall_quality_score"].mean(), 2)

    if dataset_quality_score >= 90:
        dataset_quality_status = "Excellent"
    elif dataset_quality_score >= 75:
        dataset_quality_status = "Good"
    elif dataset_quality_score >= 60:
        dataset_quality_status = "Moderate"
    else:
        dataset_quality_status = "Poor"

    high_sensitive_columns = len(
        privacy_report[privacy_report["sensitivity_level"] == "High"]
    )

    masked_columns = len(
        privacy_report[privacy_report["masking_action"].isin(["Full Mask", "Partial Mask"])]
    )

    preserved_columns = len(
        privacy_report[privacy_report["masking_action"] == "Preserve"]
    )

    summary_report.append({
        "dataset_name": dataset_name,
        "original_rows": original_rows,
        "original_columns": original_cols,
        "cleaned_rows": cleaned_df.shape[0],
        "cleaned_columns": cleaned_df.shape[1],
        "duplicate_rows_removed": duplicate_count,
        "dataset_quality_score": dataset_quality_score,
        "dataset_quality_status": dataset_quality_status,
        "high_sensitive_columns": high_sensitive_columns,
        "masked_columns": masked_columns,
        "preserved_columns": preserved_columns
    })

    print(f"Completed: {dataset_name}")

StatementMeta(, 68b2d141-3696-4c17-9191-d3978c91bd15, 17, Finished, Available, Finished, False)


Processing dataset: customer
Completed: customer

Processing dataset: billing
Completed: billing

Processing dataset: service
Completed: service


## **Generate and Display Reports**

In [10]:
final_quality_report = pd.concat(all_quality_reports, ignore_index=True)
final_privacy_report = pd.concat(all_privacy_reports, ignore_index=True)
final_summary_report = pd.DataFrame(summary_report)

dataset_quality_scores = final_quality_report.groupby("dataset_name").agg({
    "completeness_score": "mean",
    "validity_score": "mean",
    "uniqueness_score": "mean",
    "overall_quality_score": "mean"
}).reset_index().round(2)

dataset_quality_scores["dataset_quality_status"] = dataset_quality_scores["overall_quality_score"].apply(
    lambda x: "Excellent" if x >= 90 else
              "Good" if x >= 75 else
              "Moderate" if x >= 60 else
              "Poor"
)

display(final_summary_report)
display(dataset_quality_scores)
display(final_quality_report)
display(final_privacy_report)

StatementMeta(, 68b2d141-3696-4c17-9191-d3978c91bd15, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5881a7d8-6cfe-453c-b211-4cd59c4d660a)

SynapseWidget(Synapse.DataFrame, 5c3994ec-8b45-4e9c-a6a1-f3d3e10713a8)

SynapseWidget(Synapse.DataFrame, 5eb70eb2-c89d-4a74-837d-02a65a31cdaa)

SynapseWidget(Synapse.DataFrame, 6ebbd30c-4e95-4389-a45d-1edc734f6bb8)

## **Preview Cleaned and Masked datasets**

In [11]:
for dataset_name in datasets.keys():
    print(f"\n=== Cleaned Dataset Preview: {dataset_name} ===")
    display(cleaned_datasets[dataset_name].head())

    print(f"\n=== Masked Dataset Preview: {dataset_name} ===")
    display(masked_datasets[dataset_name].head())

StatementMeta(, 68b2d141-3696-4c17-9191-d3978c91bd15, 19, Finished, Available, Finished, False)


=== Cleaned Dataset Preview: customer ===


SynapseWidget(Synapse.DataFrame, 169570a6-b50d-4241-9751-1b4bca83fcb3)


=== Masked Dataset Preview: customer ===


SynapseWidget(Synapse.DataFrame, 98a66b6a-8769-4d95-a41f-c74c37462d9b)

...

SynapseWidget(Synapse.DataFrame, d749ed85-ea29-4a87-a7d9-b08613021b60)


=== Cleaned Dataset Preview: service ===


SynapseWidget(Synapse.DataFrame, 25b86d62-50bd-4aa5-a627-d81d70754134)


=== Masked Dataset Preview: service ===


SynapseWidget(Synapse.DataFrame, 92b16740-eb5e-4e05-ac2f-33712ce1a2b0)

## **Save Processed Outputs to Lakehouse Files**

In [12]:
output_base = f"/lakehouse/default/Files/processed/{data_owner}/"

folders = [
    "cleaned",
    "masked",
    "reports"
]

for folder in folders:
    os.makedirs(output_base + folder, exist_ok=True)

for dataset_name, df in cleaned_datasets.items():
    df.to_csv(
        output_base + f"cleaned/{dataset_name}_cleaned.csv",
        index=False
    )

for dataset_name, df in masked_datasets.items():
    df.to_csv(
        output_base + f"masked/{dataset_name}_masked.csv",
        index=False
    )

final_quality_report.to_csv(
    output_base + "reports/column_level_quality_report.csv",
    index=False
)

dataset_quality_scores.to_csv(
    output_base + "reports/dataset_quality_scores.csv",
    index=False
)

final_privacy_report.to_csv(
    output_base + "reports/privacy_and_masking_policy_report.csv",
    index=False
)

final_summary_report.to_csv(
    output_base + "reports/processing_summary_report.csv",
    index=False
)

print("All outputs saved to Lakehouse Files successfully.")

StatementMeta(, 68b2d141-3696-4c17-9191-d3978c91bd15, 20, Finished, Available, Finished, False)

All outputs saved to Lakehouse Files successfully.
